In [1]:
# notebooks/exploration.ipynb
import os
import sys
import yaml
import numpy as np
from nuscenes.nuscenes import NuScenes
import k3d # For K3D specific visualizations

# Add project root to sys.path
notebook_path = os.path.abspath(os.getcwd())
project_root = os.path.dirname(notebook_path)
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to sys.path: {project_root}")

%load_ext autoreload
%autoreload 2

from src.core import MDetector, OcclusionResult
from src.data_utils.nuscenes_helper import get_lidar_sweep_data
# from src.utils.visualization import plot_occlusion_test_results_k3d # If you refactor K3D plot
# from scripts.generate_mdetector_bev_video import generate_video # If you want to run video from notebook

# --- Load Config and Initialize NuScenes ---
config_file_path = os.path.join(project_root, 'config/m_detector_config.yaml')
with open(config_file_path, 'r') as f:
    config = yaml.safe_load(f)

nusc = NuScenes(version=config['nuscenes']['version'],
                dataroot=config['nuscenes']['dataroot'],
                verbose=False)

m_detector = MDetector(config=config)

scene_token = nusc.scene[1]['token'] # Example: first scene
current_sample_token = nusc.get('scene', scene_token)['first_sample_token']
lidar_sensor_name = config['nuscenes']['lidar_sensor_name']

# Determine how many sweeps to process for populating and labeling
num_sweeps_to_populate = config['initialization']['num_initial_sweeps_for_map'] + 10 # e.g., 5 for init + 10 more


Added project root to sys.path: /home/drugge/Unsupervised-Moving-Point-Detection/m_detector_python


In [2]:

print(f"Populating M-Detector library with {num_sweeps_to_populate} sweeps...")
processed_sweeps_count = 0
temp_lidar_token = current_sample_token # Use a temp token for this loop
while temp_lidar_token and processed_sweeps_count < num_sweeps_to_populate:
    print(f"Processing sweep: {processed_sweeps_count}")
    sample = nusc.get('sample', temp_lidar_token)
    lidar_token = sample['data'][lidar_sensor_name]
    points_lidar_frame, T_global_lidar, _ = get_lidar_sweep_data(nusc, lidar_token)
    lidar_sample_data_for_ts = nusc.get('sample_data', lidar_token)
    m_detector.add_sweep_and_create_depth_image(
        points_lidar_frame, T_global_lidar, lidar_sample_data_for_ts['timestamp']
    )
    processed_sweeps_count += 1
    if not sample['next']: break
    temp_lidar_token = sample['next']
print(f"M-Detector library populated with {len(m_detector.depth_image_library)} DIs.")



Populating M-Detector library with 15 sweeps...
Processing sweep: 0
Processing sweep: 1
Processing sweep: 2
Processing sweep: 3
Processing sweep: 4
Processing sweep: 5
Processing sweep: 6
Processing sweep: 7
Processing sweep: 8
Processing sweep: 9
Processing sweep: 10
Processing sweep: 11
Processing sweep: 12
Processing sweep: 13
Processing sweep: 14
M-Detector library populated with 15 DIs.


In [3]:
# --- 2. Process and Assign Labels to Points in DepthImages (Now includes Map Consistency) ---
print("\n--- Processing and Assigning Labels (Occlusion + Map Consistency) ---")
if m_detector.is_ready_for_processing():
    library = m_detector.depth_image_library
    historical_lag_indices = config.get('m_detector_processing', {}).get('historical_lag_indices', 1) # From your previous setup
    
    num_dis_processed_for_labels = 0
    # Start processing from the first DI that has a valid historical reference
    start_processing_idx = historical_lag_indices 
    
    # Use tqdm for progress if processing many DIs
    from tqdm.notebook import tqdm as tqdm_notebook # For notebook environment
    
    for i in tqdm_notebook(range(start_processing_idx, len(library)), desc="Labeling DIs"):
        current_di_to_label = library.get_image_by_index(i)
        historical_ref_di = library.get_image_by_index(i - historical_lag_indices)
        
        if current_di_to_label: # historical_ref_di can be None, process_and_label_di handles it
            # print(f"Labeling DI {i} (TS: {current_di_to_label.timestamp/1e6:.2f}s)")
            labeled_count = m_detector.process_and_label_di(current_di_to_label, historical_ref_di)
            # print(f"  Labeled {labeled_count} points in DI {i}.")
            num_dis_processed_for_labels +=1
            
    print(f"Label processing complete for {num_dis_processed_for_labels} DIs.")
else:
    print("M-Detector not ready for processing labels.")


--- Processing and Assigning Labels (Occlusion + Map Consistency) ---


Labeling DIs:   0%|          | 0/14 [00:00<?, ?it/s]

Label processing complete for 14 DIs.


In [4]:
# --- 3. Visualize a Specific DepthImage with its Processed Labels ---
from src.utils.visualization import plot_predictions_k3d 

if len(m_detector.depth_image_library) > start_processing_idx:
    index_to_plot_labels = len(m_detector.depth_image_library) - 1 # Plot the latest processed DI
    # Or choose a specific index: index_to_plot_labels = start_processing_idx 
    
    depth_image_for_viz = m_detector.depth_image_library.get_image_by_index(index_to_plot_labels)

    if depth_image_for_viz:
        print(f"\n--- K3D plot for Labeled DI at library index {index_to_plot_labels} ---")
        plot = plot_predictions_k3d(
            depth_image_with_labels=depth_image_for_viz,
            point_size=0.08
        )
        if plot:
            plot.display()


--- K3D plot for Labeled DI at library index 14 ---
Plotting 26415 points from DI (Timestamp: 1533151610.45s).
Label Summary:
  - EMPTY_IN_IMAGE: 282
  - OCCLUDED_BY_IMAGE: 676
  - OCCLUDING_IMAGE: 469
  - UNDETERMINED: 24988


/home/drugge/miniconda3/envs/mdet_env/lib/python3.7/site-packages/traittypes/traittypes.py:101: UserWarning: Given trait value dtype "float64" does not match required type "float32". A coerced copy has been created.
  np.dtype(self.dtype).name))


Output()